# Laborator 6 - Inferenta bayesiana

Vrem sa estimam $\lambda$, numarul mediu de apeluri pe ora.

Datele: am observat $10$ ore, cu $180$ de apeluri in total.

- Verosimilitate: $X_i \mid \lambda \sim \text{Poisson}(\lambda)$
- Prior: $\lambda \sim \text{Gamma}(a_0, b_0)$

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import arviz as az

ore = 10            # numarul de ore observate
suma_apeluri = 180  # total apeluri

## Priorul
Nu cunoastem rata exacta, asa ca alegem un prior larg, slab informativ: $a_0 = 2,\ b_0 = 0.1$.

- media: $a_0 / b_0 = 20$
- varianta: $a_0 / b_0^2 = 200$ (mare)

Varianta mare inseamna ca priorul influenteaza putin rezultatul, iar datele conteaza mai mult.

In [ ]:
a0, b0 = 2, 0.1

## Posteriorul
$$p(\lambda \mid x) \propto p(x \mid \lambda)\,p(\lambda) \propto \lambda^{a_0 + \sum x_i - 1}\, e^{-(b_0 + n)\lambda}$$

Aceasta este forma unei distributii Gamma, deci:
$$\lambda \mid x \sim \text{Gamma}(a_0 + \textstyle\sum x_i,\ b_0 + n) = \text{Gamma}(182,\ 10.1)$$

- media: $182 / 10.1 \approx 18.02$
- modul (valoarea cea mai probabila): $181 / 10.1 \approx 17.92$

In [ ]:
a1 = a0 + suma_apeluri
b1 = b0 + ore

print(f"posterior: Gamma({a1}, {b1})")
print(f"media: {a1 / b1:.2f}")
print(f"modul: {(a1 - 1) / b1:.2f}")

In [ ]:
# scipy foloseste scale = 1/rata
prior_dist = stats.gamma(a=a0, scale=1 / b0)
post_dist = stats.gamma(a=a1, scale=1 / b1)

grid = np.linspace(0, 60, 1000)
plt.plot(grid, prior_dist.pdf(grid), label='prior', color='gray')
plt.plot(grid, post_dist.pdf(grid), label='posterior', color='green')
plt.axvline(suma_apeluri / ore, color='red', linestyle='--', label='media observata (18)')
plt.xlabel('λ')
plt.legend()
plt.show()

Priorul este foarte larg, iar posteriorul se concentreaza in jurul lui 18. Datele domina rezultatul.

## Intervalul HDI 94% si modul

In [ ]:
sample = post_dist.rvs(size=100000, random_state=12)

print("HDI 94%:", az.hdi(sample, hdi_prob=0.94))

az.plot_posterior(sample, hdi_prob=0.94, point_estimate='mode', round_to=3)
plt.show()

## Concluzie
- Posteriorul este $\text{Gamma}(182,\ 10.1)$.
- Valoarea cea mai probabila este $\lambda \approx 17.92$ apeluri/ora. ArviZ afiseaza ~18.1 deoarece estimeaza modul din esantion, nu analitic.
- HDI 94% este aproximativ $[15.5,\ 20.5]$: cu probabilitate 94%, rata reala se afla intre 15.5 si 20.5 apeluri pe ora.